# TeClass Live Demo

Classify a Telugu (headline, article) pair as HREL / MREL / LREL using three systems:

1. **Classical ML**  TF-IDF + Logistic Regression
2. **Fine-tuned BERT**  mDeBERTa
3. **Zero-shot LLM**  Sarvam-105b with the anti-bias prompt

## 1. Setup

In [1]:
!pip install -q transformers accelerate sentencepiece torch requests scikit-learn pandas

In [2]:
import os, re, json, time
import pandas as pd
import numpy as np
import requests
import torch
from pathlib import Path

try:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    SARVAM_KEY = userdata.get('SARVAM_API')
except Exception:
    SARVAM_KEY = os.environ.get('SARVAM_API', 'PASTE_SARVAM_KEY_HERE')

LABELS = ['HREL', 'MREL', 'LREL']
lab2id = {'HREL': 0, 'MREL': 1, 'LREL': 2}
id2lab = {v: k for k, v in lab2id.items()}

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

Mounted at /content/drive
Device: cuda


## 2. Load the training data

In [6]:
TRAIN_CSV = 'TeClass_training_dataset1 - TeClass_training_data.csv'
TEST_CSV  = 'TeClass_test_dataset1 - TeClass_testing_data.csv'

for p in [TRAIN_CSV, TEST_CSV]:
    assert Path(p).exists(), f'Missing: {p}'

def load(path):
    df = pd.read_csv(path)
    df['label'] = df['label'].astype(str).str.upper().str.strip()
    df = df[df['label'].isin(LABELS)].reset_index(drop=True)
    return df

train_df = load(TRAIN_CSV)
test_df = load(TEST_CSV)
print(f'Train: {len(train_df)}   Test: {len(test_df)}')

Train: 18324   Test: 3927


## 3. Train the ML baseline  (TF-IDF + Logistic Regression)

In [35]:
!pip install -q sentencepiece
import sentencepiece as spm
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from scipy.sparse import hstack

print('Preparing SentencePiece training corpus...')
corpus_path = '/content/sp_corpus.txt'
with open(corpus_path, 'w', encoding='utf-8') as f:
    for t in train_df['article']:
        f.write(str(t) + '\n')
    for t in train_df['headline']:
        f.write(str(t) + '\n')

SP_PREFIX = '/content/teclass_sp'
SP_VOCAB_SIZE = 2945

if not os.path.exists(f'{SP_PREFIX}.model'):
    print(f'Training SentencePiece BPE (vocab {SP_VOCAB_SIZE})...')
    spm.SentencePieceTrainer.Train(
        input=corpus_path,
        model_prefix=SP_PREFIX,
        vocab_size=SP_VOCAB_SIZE,
        model_type='bpe',
        character_coverage=1.0,
        shuffle_input_sentence=False,
    )
else:
    print('SentencePiece model already trained, reusing.')

sp = spm.SentencePieceProcessor(model_file=f'{SP_PREFIX}.model')
print(f'Vocabulary size: {sp.get_piece_size()}')
print(f'Example: {sp.encode_as_pieces("తైవాన్‌లో భూకంపం")}')

def sp_tokenize(text):
    return sp.encode_as_pieces(str(text))

print('\nFitting TF-IDF with SentencePiece subwords...')
vec_art  = TfidfVectorizer(tokenizer=sp_tokenize, lowercase=False,
                           ngram_range=(1, 1), min_df=2, max_features=30000)
vec_head = TfidfVectorizer(tokenizer=sp_tokenize, lowercase=False,
                           ngram_range=(1, 1), min_df=2, max_features=10000)

Xa = vec_art.fit_transform(train_df['article'])
Xh = vec_head.fit_transform(train_df['headline'])
X = hstack([Xa, Xh]).tocsr()
y = train_df['label'].map(lab2id).values

print(f'Article TF-IDF vocab: {len(vec_art.vocabulary_)}')
print(f'Headline TF-IDF vocab: {len(vec_head.vocabulary_)}')

print('\nFitting Logistic Regression...')
ml_model = LogisticRegression(C=1.0, max_iter=2000, solver='liblinear')
ml_model.fit(X, y)
print('ML baseline trained (SentencePiece BPE + TF-IDF + LR).')

Preparing SentencePiece training corpus...
Training SentencePiece BPE (vocab 2945)...
Vocabulary size: 2945
Example: ['▁త', 'ై', 'వా', 'న్', '▁లో', '▁భూ', 'క', 'ంప', 'ం']

Fitting TF-IDF with SentencePiece subwords...


/usr/local/lib/python3.13/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Article TF-IDF vocab: 2916
Headline TF-IDF vocab: 2721

Fitting Logistic Regression...
ML baseline trained (SentencePiece BPE + TF-IDF + LR).


## 4. Load the fine-tuned mDeBERTa

In [36]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MDEBERTA_DIR = '/content/drive/MyDrive/final_model'

assert Path(MDEBERTA_DIR).exists(), f'Missing: {MDEBERTA_DIR}'

print('Loading mDeBERTa from Drive...')
bert_tok = AutoTokenizer.from_pretrained(MDEBERTA_DIR, use_fast=True)
bert_model = AutoModelForSequenceClassification.from_pretrained(MDEBERTA_DIR).to(DEVICE).eval()


print('mDeBERTa loaded.')
print('Label mapping:', bert_model.config.id2label)

Loading mDeBERTa from Drive...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

mDeBERTa loaded.
Label mapping: {0: 'HREL', 1: 'MREL', 2: 'LREL'}


## 5. Set up the Sarvam API wrapper  (anti-bias zero-shot prompt)

In [47]:
SARVAM_URL = 'https://api.sarvam.ai/v1/chat/completions'
SARVAM_MODEL = 'sarvam-105b-conversations'

DEFS = (
    'IMPORTANT: Classes are approximately balanced in the test set '
    '(roughly 1/3 HREL, 1/3 MREL, 1/3 LREL). Do NOT default to HREL. '
    'Look carefully for LREL cues like sensational language or claims unsupported by the article.\n\n'
    'Categories:\n'
    '- HREL: headline is a faithful summary of the article\u2019s main event.\n'
    '- MREL: headline is loosely related, covers a side event or weak inference.\n'
    '- LREL: headline is misleading, sensational, clickbait, or draws unsupported conclusions.'
)
LABEL_RE = re.compile(r'\b(HREL|MREL|LREL)\b', re.IGNORECASE)

def cut(text, n=2200):
    text = str(text)
    return text if len(text) <= n else text[:n] + ' [...]'

def sarvam_prompt(article, headline):
    return (
        f'Classify the relevance between a Telugu news article and headline.\n\n'
        f'{DEFS}\n\n'
        f'Now classify:\nArticle: {cut(article)}\n\nHeadline: {headline}\n\n'
        f'Reply with ONE label only: HREL, MREL, or LREL.'
    )

def call_sarvam(prompt):
    r = requests.post(
        SARVAM_URL,
        headers={'api-subscription-key': SARVAM_KEY, 'Content-Type': 'application/json'},
        json={
            'model': SARVAM_MODEL,
            'messages': [{'role': 'user', 'content': prompt}],
            'temperature': 0,
            'max_tokens': 100,
        },
        timeout=60,
    )
    r.raise_for_status()
    return r.json()['choices'][0]['message']['content'] or ''

def parse_label(text):
    if not text:
        return 'MREL'
    m = LABEL_RE.search(text)
    return m.group(1).upper() if m else 'MREL'

## 6. predict function

In [48]:
def predict_ml(headline, article):
    """Classical ML pipeline with step-by-step detail."""
    print('\n' + '-' * 76)
    print('SYSTEM 1 of 3: Classical ML   (TF-IDF + Logistic Regression)')
    print('-' * 76)

    print('\n[Step 1] Word tokenization (whitespace split, matches TF-IDF analyzer)')
    head_tokens = headline.split()
    art_tokens = article.split()
    print(f'  Headline tokens ({len(head_tokens)} total):')
    print(f'    {head_tokens[:12]}')
    print(f'  Article tokens ({len(art_tokens)} total, showing first 15):')
    print(f'    {art_tokens[:15]}')

    print('\n[Step 2] TF-IDF vectorization')
    xa = vec_art.transform([article])
    xh = vec_head.transform([headline])
    x = hstack([xa, xh]).tocsr()
    print(f'  Article   vector: shape {xa.shape}, {xa.nnz} non-zero entries')
    print(f'  Headline  vector: shape {xh.shape}, {xh.nnz} non-zero entries')
    print(f'  Combined  vector (article || headline): shape {x.shape}, {x.nnz} non-zero')

    print('\n[Step 3] Top 5 highest-weighted tokens for this article')
    art_names = vec_art.get_feature_names_out()
    xa_dense = xa.toarray()[0]
    top_idx = np.argsort(xa_dense)[::-1][:5]
    for idx in top_idx:
        if xa_dense[idx] > 0:
            print(f'    {art_names[idx]!r:40}  tfidf = {xa_dense[idx]:.3f}')

    print('\n[Step 4] Logistic Regression forward pass')
    logits = ml_model.decision_function(x)[0]
    probs = ml_model.predict_proba(x)[0]
    print(f'  Raw logits (W . x + b):')
    for i, lab_name in enumerate(LABELS):
        print(f'    {lab_name}: {logits[i]:+.3f}')
    print(f'  After softmax (class probabilities):')
    for i, lab_name in enumerate(LABELS):
        print(f'    {lab_name}: {probs[i]:.3f}')
    pred_i = int(np.argmax(probs))
    print(f'\n  >>> ML prediction: {id2lab[pred_i]}   confidence {probs[pred_i]:.2%}')
    return id2lab[pred_i], float(probs[pred_i])


def predict_bert(headline, article):
    """mDeBERTa pipeline with step-by-step detail."""
    print('\n' + '-' * 76)
    print('SYSTEM 2 of 3: Fine-tuned mDeBERTa   (278M parameters)')
    print('-' * 76)

    print('\n[Step 1] Subword tokenization (SentencePiece BPE)')
    enc = bert_tok(headline, article, truncation=True, max_length=512,
                   return_tensors='pt').to(DEVICE)
    tokens_list = bert_tok.convert_ids_to_tokens(enc['input_ids'][0].tolist())
    print(f'  Input layout: [CLS] <headline subwords> [SEP] <article subwords> [SEP]')
    print(f'  Total tokens: {len(tokens_list)}  (max 512, truncated if longer)')
    print(f'  First 15 tokens:')
    print(f'    {tokens_list[:15]}')

    print('\n[Step 2] Model input tensors')
    print(f'  input_ids       shape = {tuple(enc["input_ids"].shape)}')
    print(f'  attention_mask  shape = {tuple(enc["attention_mask"].shape)}')
    print(f'  Running on device: {DEVICE}')

    print('\n[Step 3] Forward pass through 12 transformer layers')
    with torch.no_grad():
        out = bert_model(**enc)
    logits = out.logits[0].cpu().numpy()
    probs = torch.softmax(out.logits[0], dim=-1).cpu().numpy()
    model_labels = [bert_model.config.id2label.get(i, LABELS[i]) for i in range(3)]

    print(f'  Raw logits from classification head:')
    for i, lab_name in enumerate(model_labels):
        print(f'    {lab_name}: {logits[i]:+.3f}')
    print(f'  After softmax:')
    for i, lab_name in enumerate(model_labels):
        print(f'    {lab_name}: {probs[i]:.3f}')
    pred_i = int(np.argmax(probs))
    pred_lab = model_labels[pred_i] if model_labels[pred_i] in LABELS else id2lab[pred_i]
    print(f'\n  >>> mDeBERTa prediction: {pred_lab}   confidence {probs[pred_i]:.2%}')
    return pred_lab, float(probs[pred_i])


def predict_llm(headline, article):
    """Sarvam LLM pipeline with step-by-step detail."""
    print('\n' + '-' * 76)
    print('SYSTEM 3 of 3: Zero-shot LLM   (Sarvam-105b with anti-bias prompt)')
    print('-' * 76)

    print('\n[Step 1] Build the prompt (anti-bias zero-shot template)')
    prompt = sarvam_prompt(article, headline)
    print(f'  Prompt length: {len(prompt)} characters')
    print(f'  Preview (first 500 chars):')
    print('  ' + prompt[:500].replace(chr(10), chr(10) + '  '))
    print(f'  ...')

    print('\n[Step 2] Send HTTP request to Sarvam API')
    print(f'  Endpoint:    {SARVAM_URL}')
    print(f'  Model:       {SARVAM_MODEL}')
    print(f'  Temperature: 0  (deterministic)')
    print(f'  Max tokens:  100')
    print(f'  (waiting for response...)')

    raw = call_sarvam(prompt)

    print(f'\n[Step 3] Raw response from Sarvam')
    print(f'  {raw.strip()[:300]!r}')

    print(f'\n[Step 4] Parse HREL / MREL / LREL label from response')
    label = parse_label(raw)
    print(f'  Matched token: {label!r}')
    print(f'\n  >>> Sarvam prediction: {label}')
    return label, raw.strip()[:60]


def demo_classify(headline, article, true_label=None):
    """Run all three systems with step-by-step detail and compare."""
    print()
    print('=' * 76)
    print('INPUT PAIR')
    print('=' * 76)
    print(f'Headline ({len(headline)} chars, {len(headline.split())} words):')
    print(f'  {headline}')
    print()
    disp = article[:260] + ('...' if len(article) > 260 else '')
    print(f'Article  ({len(article)} chars, {len(article.split())} words):')
    print(f'  {disp}')
    if true_label:
        print(f'\nTrue label (gold annotation): {true_label}')

    ml_lab, ml_conf = predict_ml(headline, article)
    bert_lab, bert_conf = predict_bert(headline, article)
    try:
        llm_lab, llm_raw = predict_llm(headline, article)
    except Exception as e:
        llm_lab = None
        print(f'\n  >>> LLM API error: {type(e).__name__}: {e}')

    print()
    print('=' * 76)
    print('FINAL COMPARISON')
    print('=' * 76)
    print(f'{"System":<28}{"Prediction":<12}{"Confidence":<14}{"Match":<10}')
    print('-' * 76)
    rows = [
        ('Classical ML (LR)',   ml_lab,   f'{ml_conf:.2%}'),
        ('Fine-tuned mDeBERTa', bert_lab, f'{bert_conf:.2%}'),
        ('Sarvam-105b LLM',     llm_lab or 'ERROR',  'n/a'),
    ]
    for name, lab, conf in rows:
        match = ''
        if true_label and lab:
            match = 'correct' if lab == true_label else 'wrong'
        print(f'{name:<28}{str(lab):<12}{conf:<14}{match:<10}')
    if true_label:
        print(f'\nGold label: {true_label}')
    print('=' * 76)
    print()

## 7. Run on three pre-selected test examples

In [49]:
def show_tokenization(sentence):
    """Show how a Telugu sentence gets tokenized by both pipelines."""
    print('=' * 70)
    print('INPUT SENTENCE')
    print('=' * 70)
    print(sentence)
    print(f'\nCharacters: {len(sentence)}   Words: {len(sentence.split())}')
    print()

    print('=' * 70)
    print('STEP 1: Word tokenization  (used by TF-IDF + Logistic Regression)')
    print('=' * 70)
    print('Split by whitespace and punctuation.')
    print()
    word_tokens = sentence.split()
    for i, tok in enumerate(word_tokens, 1):
        print(f'  Token {i:2d}: {tok}')
    print(f'\n  Total word tokens: {len(word_tokens)}')
    print()

    print('=' * 70)
    print('STEP 2: SentencePiece BPE subword tokenization  (used by mDeBERTa)')
    print('=' * 70)
    print('Each word is split into subword units learned from pretraining.')
    print('The "_" (shown as ▁) marks the start of a new word.')
    print()
    subwords = bert_tok.tokenize(sentence)
    for i, sw in enumerate(subwords, 1):
        print(f'  Subword {i:2d}: {sw}')
    print(f'\n  Total subword tokens: {len(subwords)}')
    print()

    print('=' * 70)
    print('STEP 3: Token IDs  (actual input to the model)')
    print('=' * 70)
    print('Each subword is replaced by its vocabulary index.')
    print()
    ids = bert_tok.convert_tokens_to_ids(subwords)
    pairs_df = pd.DataFrame({
        'Position': list(range(1, len(subwords) + 1)),
        'Subword': subwords,
        'Token ID': ids,
    })
    print(pairs_df.to_string(index=False))
    print()

    print('=' * 70)
    print('STEP 4: Full BERT input with special tokens')
    print('=' * 70)
    print('Format: [CLS] <sentence> [SEP]')
    print()
    enc = bert_tok(sentence, return_tensors='pt')
    full_tokens = bert_tok.convert_ids_to_tokens(enc['input_ids'][0].tolist())
    print(f'  Full sequence ({len(full_tokens)} tokens):')
    print(f'  {full_tokens}')
    print()
    print(f'  input_ids tensor shape: {tuple(enc["input_ids"].shape)}')
    print()

In [50]:
import pandas as pd

def predict_ml(headline, article):
    xa = vec_art.transform([article])
    xh = vec_head.transform([headline])
    x = hstack([xa, xh]).tocsr()
    probs = ml_model.predict_proba(x)[0]
    i = int(np.argmax(probs))
    return id2lab[i], float(probs[i]), probs

def predict_bert(headline, article):
    enc = bert_tok(headline, article, truncation=True, max_length=512,
                   return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        logits = bert_model(**enc).logits[0]
    probs = torch.softmax(logits, dim=-1).cpu().numpy()
    i = int(np.argmax(probs))
    model_labels = [bert_model.config.id2label.get(k, LABELS[k]) for k in range(3)]
    pred = model_labels[i] if model_labels[i] in LABELS else id2lab[i]
    return pred, float(probs[i]), probs, enc['input_ids'].shape[1]

def predict_llm(headline, article):
    prompt = sarvam_prompt(article, headline)
    raw = call_sarvam(prompt)
    return parse_label(raw), raw.strip()[:120]


def demo_classify(headline, article, true_label=None):
    print()
    print('Input')
    print('-----')
    print(f'Headline: {headline}')
    print()
    art_prev = article[:260] + ('...' if len(article) > 260 else '')
    print(f'Article:  {art_prev}')
    if true_label:
        print(f'\nTrue label: {true_label}')
    print()

    print('System 1: Classical ML (TF-IDF + Logistic Regression)')
    ml_lab, ml_conf, ml_probs = predict_ml(headline, article)
    probs_df = pd.DataFrame({
        'Class': LABELS,
        'Probability': [f'{p:.3f}' for p in ml_probs],
    })
    print(probs_df.to_string(index=False))
    print(f'Prediction: {ml_lab}  (confidence {ml_conf:.2%})')
    print()

    print('System 2: Fine-tuned mDeBERTa')
    bert_lab, bert_conf, bert_probs, ntok = predict_bert(headline, article)
    print(f'Tokenized into {ntok} subwords')
    probs_df = pd.DataFrame({
        'Class': LABELS,
        'Probability': [f'{p:.3f}' for p in bert_probs],
    })
    print(probs_df.to_string(index=False))
    print(f'Prediction: {bert_lab}  (confidence {bert_conf:.2%})')
    print()

    print('System 3: Zero-shot Sarvam-105b')
    try:
        llm_lab, llm_raw = predict_llm(headline, article)
        print(f'Raw response: {llm_raw!r}')
        print(f'Prediction: {llm_lab}')
    except Exception as e:
        llm_lab = None
        print(f'API error: {type(e).__name__}')
    print()

    print('Summary')
    print('-------')
    rows = [
        ('Classical ML', ml_lab, f'{ml_conf:.2%}'),
        ('mDeBERTa', bert_lab, f'{bert_conf:.2%}'),
        ('Sarvam-105b', llm_lab or 'error', '-'),
    ]
    if true_label:
        rows = [(name, pred, conf, 'correct' if pred == true_label else 'wrong')
                for name, pred, conf in rows]
        summary_df = pd.DataFrame(rows, columns=['System', 'Prediction', 'Confidence', 'Match'])
    else:
        summary_df = pd.DataFrame(rows, columns=['System', 'Prediction', 'Confidence'])
    print(summary_df.to_string(index=False))
    print()

In [51]:
sentence = 'తైవాన్‌లో భూకంపం'

print('=' * 70)
print('INPUT SENTENCE')
print('=' * 70)
print(sentence)
print(f'\nCharacters: {len(sentence)}   Words (whitespace): {len(sentence.split())}')
print()

print('=' * 70)
print('STEP 1: SentencePiece BPE subwords  (used by Classical ML + TF-IDF)')
print('=' * 70)
print(f'Vocabulary size: {sp.get_piece_size()}  (matches paper)')
ml_subwords = sp.encode_as_pieces(sentence)
for i, sw in enumerate(ml_subwords, 1):
    print(f'  Subword {i:2d}: {sw}')
print(f'\nTotal: {len(ml_subwords)} subword tokens')
print()

print('=' * 70)
print('STEP 2: mDeBERTa subword tokenization  (different SentencePiece model)')
print('=' * 70)
print('mDeBERTa uses its own pretrained SentencePiece tokenizer, vocab 250,002.')
bert_subwords = bert_tok.tokenize(sentence)
for i, sw in enumerate(bert_subwords, 1):
    print(f'  Subword {i:2d}: {sw}')
print(f'\nTotal: {len(bert_subwords)} subword tokens')
print()

print('=' * 70)
print('STEP 3: Token IDs for mDeBERTa input')
print('=' * 70)
ids = bert_tok.convert_tokens_to_ids(bert_subwords)
for pos, (sw, tid) in enumerate(zip(bert_subwords, ids), 1):
    print(f'  Position {pos:2d}  →  Subword: {sw:<10}  →  ID: {tid}')
print()

print('=' * 70)
print('STEP 4: Full mDeBERTa input with [CLS] and [SEP] tokens')
print('=' * 70)
enc = bert_tok(sentence, return_tensors='pt')
full_tokens = bert_tok.convert_ids_to_tokens(enc['input_ids'][0].tolist())
print(f'Full token list ({len(full_tokens)} tokens):')
print(f'  {full_tokens}')
print(f'\ninput_ids tensor shape: {tuple(enc["input_ids"].shape)}')

INPUT SENTENCE
తైవాన్‌లో భూకంపం

Characters: 16   Words (whitespace): 2

STEP 1: SentencePiece BPE subwords  (used by Classical ML + TF-IDF)
Vocabulary size: 2945  (matches paper)
  Subword  1: ▁త
  Subword  2: ై
  Subword  3: వా
  Subword  4: న్
  Subword  5: ▁లో
  Subword  6: ▁భూ
  Subword  7: క
  Subword  8: ంప
  Subword  9: ం

Total: 9 subword tokens

STEP 2: mDeBERTa subword tokenization  (different SentencePiece model)
mDeBERTa uses its own pretrained SentencePiece tokenizer, vocab 250,002.
  Subword  1: ▁త
  Subword  2: ైవ
  Subword  3: ాన్
  Subword  4: ‌
  Subword  5: లో
  Subword  6: ▁భూ
  Subword  7: కం
  Subword  8: పం

Total: 8 subword tokens

STEP 3: Token IDs for mDeBERTa input
  Position  1  →  Subword: ▁త          →  ID: 10587
  Position  2  →  Subword: ైవ          →  ID: 136054
  Position  3  →  Subword: ాన్         →  ID: 73592
  Position  4  →  Subword: ‌           →  ID: 3
  Position  5  →  Subword: లో          →  ID: 2966
  Position  6  →  Subword: ▁భూ         →  

In [52]:
demo_examples = [
    (
        'HREL',
        'తైవాన్‌లో భూకంపం',
        'తైపే : తైవాన్ భూమి కంపించింది . భూకంప తీవ్రత రిక్టర్ స్కేల్‌పై 6.4గా నమోదైంది . భారీ భూకంపం రావడంతో తైవాన్‌లో 17 అంతస్తుల భవనం కుప్పకూలింది . ఈ ఘటనలో ఇప్పటి వరకు ఐదుగురు మృతి చెందగా , భవనం శిథిలాల కింద పలువురు చిక్కుకున్నట్లు అధికారులు తెలిపారు . తైవాన సైన్యం 150 మందిని రక్షించింది . సహాయక చర్యలను తైవాన్ ప్రభుత్వం ముమ్మరంగా చేపట్టింది .',
    ),
    (
        'MREL',
        'ధావన్ ఆర్థశతకం పూర్తి , స్కోర్ - 100/0',
        'ఢిల్లీ : భారత్ - న్యూజిలాండ్‌ల మధ్య ఫిరోజ్‌షా కోట్ల మైదానంలో జరుగుతున్న తొలి టీ20లో టాస్ ఓడి బ్యాటింగ్‌కు దిగిన భారత జట్టుకు శుభారంభం లభించింది . ఓపెనర్లు రోహిత్ శర్మ , శిఖర్ ధావన్‌లు దూకుడుగా ఆడుతున్నారు . ఈ క్రమంలో ధావన్ ( 51 ) ఆర్థ శతకం పూర్తి చేశాడు . రోహిత్ ( 40 ) పరుగులు చేశారు . దీంతో 12 ఓవర్లు ముగిసే సమయానికి భారత జట్టు 100 పరుగులు చేసింది .',
    ),
    (
        'LREL',
        'పసిడి ధర పైపైకి…',
        'ఢిల్లీ : బంగారం ధరలు పెరిగాయి . మంగళవారం రూ . 255 పెరగడంతో పది గ్రాముల బంగారం ధర రూ . 30,390కి చేరింది . అంతర్జాతీయ పరిస్థితులు , పెళ్లిళ్ల సీజన్ కావడంతో స్థానిక ఆభరణాల తయారీదారుల నుంచి కొనుగోళ్లు ఊపందుకున్నాయి . దీంతో పసిడి ధర పెరిగినట్లు బులియన్ మార్కెట్ వర్గాలు వెల్లడించాయి . మరోవైపు బంగారం బాటలోనే వెండి కూడా రూ . 650లు పెరగడంతో కిలో వెండి రూ . 40,700కి చేరింది .',
    ),
]

for true_lab, head, art in demo_examples:
    demo_classify(head, art, true_label=true_lab)
    time.sleep(1)



Input
-----
Headline: తైవాన్‌లో భూకంపం

Article:  తైపే : తైవాన్ భూమి కంపించింది . భూకంప తీవ్రత రిక్టర్ స్కేల్‌పై 6.4గా నమోదైంది . భారీ భూకంపం రావడంతో తైవాన్‌లో 17 అంతస్తుల భవనం కుప్పకూలింది . ఈ ఘటనలో ఇప్పటి వరకు ఐదుగురు మృతి చెందగా , భవనం శిథిలాల కింద పలువురు చిక్కుకున్నట్లు అధికారులు తెలిపారు . తైవాన సైన్యం ...

True label: HREL

System 1: Classical ML (TF-IDF + Logistic Regression)
Class Probability
 HREL       0.724
 MREL       0.243
 LREL       0.033
Prediction: HREL  (confidence 72.41%)

System 2: Fine-tuned mDeBERTa
Tokenized into 148 subwords
Class Probability
 HREL       0.819
 MREL       0.169
 LREL       0.012
Prediction: HREL  (confidence 81.93%)

System 3: Zero-shot Sarvam-105b
Raw response: 'HREL'
Prediction: HREL

Summary
-------
      System Prediction Confidence   Match
Classical ML       HREL     72.41% correct
    mDeBERTa       HREL     81.93% correct
 Sarvam-105b       HREL          - correct


Input
-----
Headline: ధావన్ ఆర్థశతకం పూర్తి , స్కోర్ - 100/0

Article: 

In [53]:
live_headline = 'ధావన్ ఆర్థశతకం పూర్తి , స్కోర్ - 100/0'
live_article  = 'ఢిల్లీ : భారత్ - న్యూజిలాండ్‌ల మధ్య ఫిరోజ్‌షా కోట్ల మైదానంలో జరుగుతున్న తొలి టీ20లో టాస్ ఓడి బ్యాటింగ్‌కు దిగిన భారత జట్టుకు శుభారంభం లభించింది . ఓపెనర్లు రోహిత్ శర్మ , శిఖర్ ధావన్‌లు దూకుడుగా ఆడుతున్నారు . ఈ క్రమంలో ధావన్ ( 51 ) ఆర్థ శతకం పూర్తి చేశాడు . రోహిత్ ( 40 ) పరుగులు చేశారు . దీంతో 12 ఓవర్లు ముగిసే సమయానికి భారత జట్టు 100 పరుగులు చేసింది .'

if live_headline.strip() and live_article.strip():
    demo_classify(live_headline, live_article)
else:
    print('(empty) Paste a Telugu headline and article into the two strings above, then run again.')


Input
-----
Headline: ధావన్ ఆర్థశతకం పూర్తి , స్కోర్ - 100/0

Article:  ఢిల్లీ : భారత్ - న్యూజిలాండ్‌ల మధ్య ఫిరోజ్‌షా కోట్ల మైదానంలో జరుగుతున్న తొలి టీ20లో టాస్ ఓడి బ్యాటింగ్‌కు దిగిన భారత జట్టుకు శుభారంభం లభించింది . ఓపెనర్లు రోహిత్ శర్మ , శిఖర్ ధావన్‌లు దూకుడుగా ఆడుతున్నారు . ఈ క్రమంలో ధావన్ ( 51 ) ఆర్థ శతకం పూర్తి చేశాడు . రోహి...

System 1: Classical ML (TF-IDF + Logistic Regression)
Class Probability
 HREL       0.342
 MREL       0.630
 LREL       0.028
Prediction: MREL  (confidence 63.00%)

System 2: Fine-tuned mDeBERTa
Tokenized into 166 subwords
Class Probability
 HREL       0.229
 MREL       0.733
 LREL       0.038
Prediction: MREL  (confidence 73.26%)

System 3: Zero-shot Sarvam-105b
Raw response: 'HREL'
Prediction: HREL

Summary
-------
      System Prediction Confidence
Classical ML       MREL     63.00%
    mDeBERTa       MREL     73.26%
 Sarvam-105b       HREL          -



## 8. pick a random real test row

In [54]:
row = test_df.sample(1, random_state=None).iloc[0]
demo_classify(row['headline'], row['article'], true_label=row['label'])


Input
-----
Headline: అంత గొప్పగా నటించడ కష్టం : మాధురీ దీక్షిత్

Article:  ఓ నాయిక చేయాల్సిన సినిమా మరో తారకు వెళ్లడం సహజం . ఆ ఇద్దరు నాయికల్లో ఒకరు భౌతికంగా దూరమయితే ఆ పాత్రలో నటించే మరో తారకు ఎంతో ఇబ్బందిగా ఉంటుంది . బాలీవుడ్ నాయిక మాధురీ దీక్షిత్ ఇలాంటి సందర్భాన్నే ఎదుర్కొంటోంది . దర్శక నిర్మాత కరణ్ జోహార్ తన కలల చిత్రంగా కలంక్ అన...

True label: MREL

System 1: Classical ML (TF-IDF + Logistic Regression)
Class Probability
 HREL       0.044
 MREL       0.687
 LREL       0.269
Prediction: MREL  (confidence 68.67%)

System 2: Fine-tuned mDeBERTa
Tokenized into 383 subwords
Class Probability
 HREL       0.077
 MREL       0.796
 LREL       0.127
Prediction: MREL  (confidence 79.61%)

System 3: Zero-shot Sarvam-105b
Raw response: 'MREL'
Prediction: MREL

Summary
-------
      System Prediction Confidence   Match
Classical ML       MREL     68.67% correct
    mDeBERTa       MREL     79.61% correct
 Sarvam-105b       MREL          - correct

